# 02. EDA и ABC/XYZ-сегментация

Ноутбук анализирует дневную витрину `sales_date × stock_code × market_id`, показывает распределения спроса и выручки, оценивает долю возвратов и строит ABC/XYZ-сегментацию.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, RESULTS_DIR
from src.segmentation import build_abc_xyz_segments

In [ ]:
mart_daily_sales = pd.read_csv(PROCESSED_DATA_DIR / 'mart_daily_sales.csv', parse_dates=['sales_date'])
mart_daily_sales.head()

## Распределение чистого спроса

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(mart_daily_sales['net_sales_qty'], bins=50, ax=ax)
ax.set_title('Распределение net_sales_qty')
ax.set_xlabel('Чистое количество продаж')
ax.set_ylabel('Число строк')
plt.tight_layout()

## Распределение выручки

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(mart_daily_sales['revenue'], bins=50, ax=ax)
ax.set_title('Распределение revenue')
ax.set_xlabel('Выручка')
ax.set_ylabel('Число строк')
plt.tight_layout()

## Доля возвратов

In [ ]:
returns_share = mart_daily_sales['is_return_flag'].mean()
returns_summary = pd.DataFrame({'metric': ['returns_share'], 'value': [returns_share]})
returns_summary

## Топ рынков по выручке

In [ ]:
top_markets = (
    mart_daily_sales.groupby('market_id', as_index=False)
    .agg(revenue=('revenue', 'sum'), net_sales_qty=('net_sales_qty', 'sum'))
    .sort_values('revenue', ascending=False)
    .head(10)
)
top_markets

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=top_markets, x='revenue', y='market_id', ax=ax)
ax.set_title('Топ рынков по выручке')
ax.set_xlabel('Выручка')
ax.set_ylabel('Рынок')
plt.tight_layout()

## ABC/XYZ-сегментация

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
segments = build_abc_xyz_segments(mart_daily_sales, RESULTS_DIR / 'abc_xyz_segments.csv')
segments.head()

In [ ]:
segment_matrix = (
    segments.groupby(['abc_segment', 'xyz_segment'], as_index=False)
    .agg(items_cnt=('stock_code', 'count'), revenue=('revenue', 'sum'))
    .sort_values(['abc_segment', 'xyz_segment'])
)
segment_matrix

## Выводы

- Доля строк с возвратами в дневной витрине: `[A]`.
- Рынки с максимальной выручкой: `[B]`.
- Сегменты AX/BX обычно удобнее для регулярного пополнения, потому что дают вклад в выручку и более стабильный спрос.
- Сегменты AZ/BZ/CZ требуют осторожного подхода: высокий разброс спроса может ухудшать прогноз и увеличивать риск ошибки в запасах.